# Heretic Colab Setup
このノートブックは Heretic を Google Colab 上で動かすための雛形です。GPU を選択し、必要な依存をインストールしてからモデルを実行します。

In [ ]:
# 最初に Google Drive をマウントします（先にマウントしておくと権限エラーを防げます）
from google.colab import drive
import os

print('Mounting Google Drive...')
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/heretic'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('Drive mounted at /content/drive. Using', DRIVE_ROOT)

In [1]:
# ランタイム確認（GPU が利用可能か確認し、なければ対処法を表示します）
import shutil, sys, textwrap
if shutil.which('nvidia-smi') is None:
    print(textwrap.dedent('''
nvidia-smi が見つかりません。GPU が有効なランタイムを選択してください。
'''))
else:
    # GPU ドライバ情報を表示
    !nvidia-smi

Sat Apr 18 07:39:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
# pip を最新化して依存を入れる（CUDA バージョンに合わせて index-url を変更してください）
!pip install pip -q
!pip install uv -q

!uv self version

uv 0.11.7 (x86_64-unknown-linux-gnu)


In [1]:
# 例: cu121 を使用する場合
!uv pip install --upgrade torch torchvision --index-url https://download.pytorch.org/whl/cu121 --no-cache -q
!uv pip install --upgrade bitsandbytes --no-cache -q
!uv pip install --upgrade transformers --no-cache -q
# !uv pip install git+https://github.com/p-e-w/heretic.git -q
# もしくはローカル repo をアップロードして uv pip install -e /path/to/repo

!uv pip list | grep -E "torch|transformers|bitsandbytes"

^C
Using Python 3.12.13 environment at: /usr
sentence-transformers                    5.4.0
torch                                    2.10.0+cpu
torchao                                  0.10.0
torchaudio                               2.10.0+cpu
torchcodec                               0.10.0
torchdata                                0.11.0
torchsummary                             1.5.1
torchtune                                0.6.1
torchvision                              0.25.0+cpu
transformers                             5.5.4


In [ ]:
# GitHub PAT を使って private リポジトリをクローンし、editable install するセル
# トークンは非表示で入力。リポジトリとブランチを入力（例: user/repo と main）
import getpass, subprocess, importlib.util, gc, os, sys

token = getpass.getpass('Enter GitHub PAT (scopes: repo or public_repo): ')
repo = input("Repo (owner/repo) [p-e-w/heretic]: ") or "p-e-w/heretic"
branch = input("Branch [main]: ") or "main"
repo_name = repo.split('/')[-1]
dest = f"/content/{repo_name}"

if not token:
    print('トークンが入力されませんでした。クローンをスキップします。')
else:
    try:
        subprocess.run(['git','-c',f'http.extraheader=AUTHORIZATION: bearer {token}','clone',f'https://github.com/{repo}.git', dest], check=True)
        # ブランチ切替（存在しなければエラーにならないように許容）
        subprocess.run(['git','-C', dest, 'checkout', branch], check=False)
        print('クローン成功:', dest)
    except subprocess.CalledProcessError as e:
        print('git clone が失敗しました:', e)
    # トークンをメモリから消去
    token = None
    gc.collect()
    # 編集可能インストール（失敗してもノートブックは継続）
    try:
        subprocess.run([sys.executable, '-m', 'pip','install','-e', dest], check=True)
    except subprocess.CalledProcessError:
        print('pip install -e に失敗しました。手動で確認してください。')
    spec = importlib.util.find_spec('heretic')
    print('heretic importable:', spec is not None)


In [ ]:
# ブランチ指定で git clone と editable install を行うウィジェット（Colab向け）
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
except Exception:
    # ipywidgets が無ければインストールしてからインポート（Colab 環境向け）
    !pip install -q ipywidgets
    import importlib
    importlib.invalidate_caches()
    import ipywidgets as widgets
    from IPython.display import display, clear_output

repo_text = widgets.Text(value='Akikukeo1/Live-Vision-Narrator', description='Repo:')
branch_text = widgets.Text(value='abliterator', description='Branch:')
dest_text = widgets.Text(value='/content/Live-Vision-Narrator', description='Dest:')
token_text = widgets.Password(description='GitHub PAT:')
out = widgets.Output(layout={'border': '1px solid #ccc'})

def on_click(b):
    with out:
        clear_output()
        repo = repo_text.value.strip() or 'Akikukeo1/Live-Vision-Narrator'
        branch = branch_text.value.strip() or 'abliterator'
        dest = dest_text.value.strip() or f'/content/{repo.split('/')[-1]}'
        token = token_text.value.strip()
        print(f'Cloning {repo} (branch {branch}) into {dest} ...')
        import os, subprocess, sys
        # 既に存在する場合は fetch/checkout を試みる
        if os.path.exists(dest):
            print('既に存在するため fetch+checkout を試みます')
            try:
                subprocess.run(['git','-C',dest,'fetch','--depth','1','origin',branch], check=True)
                subprocess.run(['git','-C',dest,'checkout',branch], check=False)
                subprocess.run(['git','-C',dest,'pull','--ff-only'], check=False)
            except subprocess.CalledProcessError as e:
                print('fetch/checkout が失敗しました:', e)
        else:
            try:
                if token:
                    env_header = f'http.extraheader=AUTHORIZATION: bearer {token}'
                    subprocess.run(['git','-c',env_header,'clone','--depth','1','--branch',branch,f'https://github.com/{repo}.git',dest], check=True)
                else:
                    subprocess.run(['git','clone','--depth','1','--branch',branch,f'https://github.com/{repo}.git',dest], check=True)
                print('クローン成功')
            except subprocess.CalledProcessError as e:
                print('git clone が失敗しました:', e)
        print('pip install -e', dest)
        try:
            subprocess.run([sys.executable,'-m','pip','install','-e',dest], check=True)
            print('pip install -e 完了')
        except subprocess.CalledProcessError:
            print('pip install -e に失敗しました。setup.py/pyproject の有無を確認してください。')
        # セキュリティのためトークンを消去
        token_text.value = ''
        print('完了')

btn = widgets.Button(description='Clone & Install', button_style='primary')
btn.on_click(on_click)
display(widgets.VBox([repo_text, branch_text, dest_text, token_text, btn, out]))


In [ ]:
# ローカルの Heretic を Colab に持ち込み、編集可能インストールするセル（Drive またはブラウザ経由）
# 方法 A: Google Drive 経由（推奨）:
#   1) ローカルで heretic リポジトリを zip にして Google Drive の任意の場所にアップロード
#   2) 以下のパスを必要に応じて書き換えて実行
from google.colab import drive
drive.mount('/content/drive')
# 例: Drive の 'MyDrive/heretic.zip' を展開して editable install
!unzip -q /content/drive/MyDrive/heretic.zip -d /content/ || true
!pip install -e /content/heretic -q || pip install -e /content/heretic --no-deps -q

# 方法 B: ブラウザからアップロード（小さめの変更向け）
from google.colab import files
uploaded = files.upload()
# アップロード後、例えば heretic.zip をアップロードした場合の処理例:
if 'heretic.zip' in uploaded:
  import os
  os.system('unzip -q heretic.zip -d /content/ || true')
  os.system('pip install -e /content/heretic -q')

# インストール確認
import importlib.util
spec = importlib.util.find_spec('heretic')
print('heretic installed:', spec is not None)

# 補足: ローカルで使っている 'uv' ラッパーは Colab 環境では不要です。Colab では直接 'pip install -e' を使ってください。

In [ ]:
# Google Drive をマウントしてモデルや結果を永続化する（必要なら）
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Heretic を実行する
# モデルは Hugging Face の model id を使うか、Drive に置いたローカルパスを渡す
!heretic --model google/gemma-4-E4B-it --device-map "auto"

## トラブルシュート
- bitsandbytes が import できない／CUDA 互換性エラー → `pip install --upgrade torch` を GPU の CUDA に合わせて再インストール
- メモリ不足でマージが失敗する → `--quantization bnb_4bit` のまま `merge` を避けるか、大きなインスタンス（Colab Pro+ 等）/別環境でマージ実行
- セッション切断が頻発する場合は Drive に中間結果を保存するスクリプトを挟む